# Nasdaq ITCH Feed Handler v1

This processig system is for the v1 release of the Nasdaq ITCH Feed Handler. In its default state, it tracks the Apple Stock using Nasdaq ITCH historical data from 30/12/2019


In [ ]:
# imports

from pynq import Overlay, allocate, MMIO
import numpy as np
import time
import gzip
import struct
import math
import golden.runner as test


In [ ]:
# Constants - should  not change unless hardware design is changed

DMA_DATA    = 0x4040
GPIO0       = 0x4120
GPIO1       = 0x4121
GPIO2       = 0x4122    
START_BYTES = 2

PATH        = "/home/xilinx/jupyter_notebooks/Nasdaq/v1release.bit"
DATA_PATH   = "/home/xilinx/jupyter_notebooks/Nasdaq/12302019.NASDAQ_ITCH50.gz" # specific historical data, also feel free to change


# Feel free to change any variables here

PATH        = "/home/xilinx/jupyter_notebooks/Nasdaq/v1release.bit"
DATA_PATH   = "/home/xilinx/jupyter_notebooks/Nasdaq/12302019.NASDAQ_ITCH50.gz" # specific historical data

SW_TARGET_SYMBOL = "AAPL" # ensure this matches the Hardware value
HW_TARGET_SYMBOL   = b"AAPL    " # This is 4 spaces after the L - required for accurate data

BASE_PRICE_STOCK_1  = 0x5208 # base price of $210.00 (for apple)
SW_MESSAGES_TO_READ = 1_000_000 # number of messages read in software




This function is used to generate network headers to pass to the system, the historical data filters these out, but we include them to allow for stripping of network headers in our system

In [ ]:
# network header gen function

def generate_network_headers(payload_len, seq_num):

    mold_len = 20 + 2 + payload_len     
    udp_len = 8 + mold_len              
    ip_total_len = 20 + udp_len          

    # Ethernet - 14 bytes of form: Dest MAC, Src MAC, EtherType (0x0800 for IPv4)
    eth = struct.pack('!6s6sH', b'\x00\x11\x22\x33\x44\x55', b'\xAA\xBB\xCC\xDD\xEE\xFF', 0x0800)

    # 2. IPv4 - 20 bytes of form: VHL, TOS, TotalLen, ID, Flags/Frag, TTL, Protocol (17=UDP), Checksum, SrcIP, DestIP
    ip = struct.pack('!BBHHHBBH4s4s',
                     0x45, 0x00, ip_total_len, 0x0000, 0x0000, 
                     64, 17, 0x0000, 
                     b'\xc0\xa8\x01\x64', b'\xc0\xa8\x01\xc8')

    # 3. UDP - 8 bytes of form: SrcPort, DestPort, Length, Checksum
    udp = struct.pack('!HHHH', 12345, 12345, udp_len, 0x0000)

    # 4. MoldUDP64 - 20 bytes of form: Session (10 bytes), Sequence Number (8 bytes), Message Count (2 bytes)
    mold = struct.pack('!10sQH', b'SESSION123', seq_num, 1)

    # 5. Mold Message Block Length 2 bytes
    msg_len_hdr = struct.pack('!H', payload_len)

    return eth + ip + udp + mold + msg_len_hdr

In [ ]:
# Overlay Load

ol = Overlay(PATH)
ol.download()
print("Overlay Loaded:")
dma = ol.axi_dma_0
gpio_bid   = ol.axi_gpio_0
gpio_ask   = ol.axi_gpio_1
gpio_base_price = ol.axi_gpio_3



## Hardware run

This wil run the hardware design, outputting data in a hw_out.txt file


In [ ]:
send_buffer = allocate(shape=(256,), dtype=np.uint32) 

with open("hw_out.txt", "w"):
    pass

target_locate   = None
gpio_base_price.channel1.write(BASE_PRICE_STOCK_1, 0xffff_ffff)
# Trackers
msg_count = 0
target_msg_count = 0
last_bid_price, last_ask_price = 0, 0
last_bid_shares, last_ask_shares = 0, 0

# opening .gz file from nasdaq itch history website
with gzip.open(DATA_PATH, "rb") as data:
    while True:
        start_msg = data.read(2) # Reading the 2-byte length header
        if not start_msg: break
            
        ins_len = int.from_bytes(start_msg, byteorder='big')
        output_data = data.read(ins_len)
        if len(output_data) < ins_len: break
            
        msg_count += 1
        msg_type = output_data[0:1]

        if msg_type == b'R' and output_data[11:15] == HW_TARGET_SYMBOL[0:4]:
            target_locate = output_data[1:3]
            print(f"Found Target! Locate: {target_locate.hex()}")

        # Message modify step for Hardware
        modified_msg = bytearray(output_data)
        keep_message = False

        if msg_type == b'S': keep_message = True
        elif target_locate and output_data[1:3] == target_locate:
            modified_msg[1:3] = (1).to_bytes(2, byteorder='big')
            # Price Division for hardware use
            offset = 32 if msg_type in [b'A', b'F', b'C'] else (31 if msg_type == b'U' else None)
            if offset:
                price = int.from_bytes(modified_msg[offset:offset+4], 'big') // 100
                modified_msg[offset:offset+4] = price.to_bytes(4, 'big')
            keep_message = True
            
            
        if not dma.sendchannel.running:
            dma.sendchannel.start()

        if keep_message:
            # Generate the 64-byte network header chain
            network_headers = generate_network_headers(payload_len=ins_len, seq_num=msg_count)
            
            full_packet = network_headers + modified_msg
            
            # Pad total packet to 4-byte boundary
            packet_len = len(full_packet)
            pad_len = math.ceil(packet_len / 4) * 4
            padded = full_packet.ljust(pad_len, b'\x00')
            
            # Big Endian swap
            temp_arr = np.frombuffer(padded, dtype=np.uint32)
            send_buffer[:len(temp_arr)] = temp_arr.byteswap()
            
            dma.sendchannel.transfer(send_buffer, nbytes=pad_len)
            dma.sendchannel.wait()

            if msg_type != b'S':
                bid_price = gpio_bid.channel1.read()
                ask_price = gpio_ask.channel1.read()
                if bid_price != last_bid_price or ask_price != last_ask_price:
                    last_bid_price, last_ask_price = bid_price, ask_price
                    last_bid_shares = gpio_bid.channel2.read()
                    last_ask_shares = gpio_ask.channel2.read()

                    bid_formatted = f"${(last_bid_price / 100.0):.2f}"
                    ask_formatted = f"${(last_ask_price / 100.0):.2f}"

                    with open("hw_out.txt", "a") as io:
                        io.write(f"{last_bid_shares:>4} Bid shares at price: {bid_formatted:>8} | "
                                 f"{last_ask_shares:>4} Ask shares at price: {ask_formatted:>8}\n")


## Software run

This run uses the golden model with the same historical data, we only read 1,000,000 of the messages (this can be altered). Relevent data in JSON form is in the file `golden_states.json`

In [ ]:
# Test with golden model

subsection = bytearray()

with gzip.open(DATA_PATH, "rb") as data:
    for _ in range(SW_MESSAGES_TO_READ):
        start_msg = data.read(2) # Reading the 2-byte length header
        if not start_msg: 
            break
            
        ins_len = int.from_bytes(start_msg, byteorder='big')
        output_data = data.read(ins_len)
        if len(output_data) < ins_len: 
            break
        
        subsection.extend(start_msg)
        subsection.extend(output_data)

print(f"Extracted {SW_MESSAGES_TO_READ} messages. Running golden model...")

with open("golden_events.jsonl", "w", encoding="utf-8") as events_out, \
     open("golden_states.jsonl", "w", encoding="utf-8") as states_out:
    
    # run_bytes natively accepts bytearray objects
    test.run_bytes(
        data=subsection,
        events_out=events_out,
        states_out=states_out,
        symbol=SW_TARGET_SYMBOL
    )

print("Subsection processing complete!")